<a href="https://colab.research.google.com/github/Cheikhcisse1/projet_FN/blob/main/SerieTemporelle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Les bibiothéques Besion

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
class SerieTemporelle:
  def __init__(self, donnee):
    self.donnee = donnee.copy()
    # On convertit et trie par date dès l'initialisation pour garantir la cohérence temporelle
    if 'date' in self.donnee.columns:
      self.donnee['date'] = pd.to_datetime(self.donnee['date'])
      self.donnee = self.donnee.sort_values('date')

  def traitementDes_donne(self):
    print("Aperçu")
    print(self.donnee.head())
    print('-----------------------'*5)
    print("Shape")
    print(self.donnee.shape)
    print('-----------------------'*5)
    print("Description")
    print(self.donnee.describe())
    print('-----------------------'*5)
    print("Info")
    print(self.donnee.info())
    print('-----------------------'*5)
    print("Valeurs manquantes")
    print(self.donnee.isnull().sum())
    print('-----------------------'*5)
    print("Valeurs dupliqués")
    print(self.donnee.duplicated().sum())
    print('-----------------------'*5)

  def visualisation(self):
    plt.figure(figsize=(10, 5))
    if 'date' in self.donnee.columns:
      plt.plot(self.donnee['date'], self.donnee['aqi'], label="AQI")
    else:
      plt.plot(self.donnee.index, self.donnee['aqi'], label="AQI")
    plt.title("L'évolution de l'AQI mensuel")
    plt.xlabel("Date")
    plt.ylabel("Valeurs AQI")
    plt.legend()
    plt.grid(True)
    plt.show()

  def traitementValeurmanquantes(self):
    # Interpolation linéaire pour maintenir la continuité temporelle
    self.donnee['aqi'] = self.donnee['aqi'].interpolate(method='linear')
    print("Valeurs manquantes après traitement (interpolation)")
    print(self.donnee.isnull().sum())
    print('-----------------------'*5)
    print("Valeurs dupliqués après traitement")
    print(self.donnee.duplicated().sum())
    print('-----------------------'*5)
    print("Shape après traitement")
    print(self.donnee.shape)
    print('-----------------------'*5)

  def preparer_index(self):
    # Méthode interne pour s'assurer que l'index est temporel et régulier
    if 'date' in self.donnee.columns:
      self.donnee.set_index('date', inplace=True)
    if not isinstance(self.donnee.index, pd.DatetimeIndex):
      self.donnee.index = pd.to_datetime(self.donnee.index)
    # .to_timestamp() sans argument replace correctement l'index au début du mois
    self.donnee.index = self.donnee.index.to_period('M').to_timestamp()

  def decomposition(self):
    self.preparer_index()
    # Décomposition additive avec une période de 12 (mensuelle)
    resultat = seasonal_decompose(self.donnee['aqi'], model='additive', period=12)

    # Visualisation
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

    ax1.plot(self.donnee['aqi'], label='Original')
    ax1.legend(loc='upper left')
    ax1.set_title('Décomposition de la série temporelle')
    ax1.grid(True)

    ax2.plot(resultat.trend, label='Trend', color='orange')
    ax2.legend(loc='upper left')
    ax2.grid(True)

    ax3.plot(resultat.seasonal, label='Seasonal', color='green')
    ax3.legend(loc='upper left')
    ax3.grid(True)

    ax4.plot(resultat.resid, label='Residual', color='red')
    ax4.legend(loc='upper left')
    ax4.grid(True)

    plt.tight_layout()
    plt.show()

    decomposition_df = pd.DataFrame({
        'aqi': self.donnee['aqi'],
        'trend': resultat.trend,
        'seasonal': resultat.seasonal,
        'residual': resultat.resid
    }, index=self.donnee.index)
    return decomposition_df

  def testSaisonnalité(self):
    self.preparer_index()
    resultat = seasonal_decompose(self.donnee['aqi'], model='additive', period=12)

    plt.figure(figsize=(10, 4))
    plt.plot(resultat.seasonal, label='Seasonal', color='green')
    plt.title('Composante Saisonnière (Saisonnalité)')
    plt.xlabel('Date')
    plt.ylabel('Valeur AQI')
    plt.legend()
    plt.grid(True)
    plt.show()

    decomposition_df = pd.DataFrame({
        'aqi': self.donnee['aqi'],
        'trend': resultat.trend,
        'seasonal': resultat.seasonal,
        'residual': resultat.resid
    }, index=self.donnee.index)
    return decomposition_df

In [ ]:
if __name__ == "__main__":
  chemin = '/content/aqi_mensuel.csv'

  # Étape 1 : Analyse exploratoire & Visualisation initiale
  donnee_base = pd.read_csv(chemin)
  donnee = SerieTemporelle(donnee_base.copy())
  donnee.traitementDes_donne()
  donnee.visualisation()

  # Étape 2 : Traitement des valeurs manquantes
  donnee.traitementValeurmanquantes()

  # Étape 3 : Décomposition et affichage
  print("la décomposition ")
  df_decomp = donnee.decomposition()
  print(df_decomp.head(12))

  print('********************************'*5)
  # Teste de saisonnalité
  print("la saisonnalité")
  df_saison = donnee.testSaisonnalité()
  print(df_saison.head(12))